In [ ]:
from urllib.request import urlopen
import pandas as pd
import numpy as np
import json, sys, os
from pymatgen.core.lattice import Lattice
from pymatgen.core.structure import Structure
from pymatgen.core import Composition
import matplotlib.pyplot as plt
import mlcpquicker
import mlcpquick
from tqdm import tqdm
import importlib
import plotly.express as px
import seaborn as sns

import reference3 as ref3
import mlcpcode2

from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, HistGradientBoostingClassifier,HistGradientBoostingRegressor,GradientBoostingClassifier,GradientBoostingRegressor
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix,mean_squared_error,mean_absolute_error,ConfusionMatrixDisplay, r2_score, f1_score,roc_curve, roc_auc_score


In [ ]:
importlib.reload(mlcpquicker)
importlib.reload(mlcpquick)
importlib.reload(ref3)
importlib.reload(mlcpcode2)

In [ ]:
#pandas view settings
pd.set_option('display.max_rows',60)
pd.set_option('display.min_rows',50)
pd.set_option('display.max_columns',None)

## LOADING DATA and making dataslices

In [ ]:
ATM = pd.read_pickle('ATM.pkl')
ATMR = pd.read_pickle('ATMR.pkl')
ATMNE = pd.read_pickle('ATMNE.pkl')
ATME = pd.read_pickle('ATME.pkl')
ATMNER = pd.read_pickle('ATMNER.pkl')
ATMER = pd.read_pickle('ATMER.pkl')
allatoms = pd.read_pickle('allatoms.pkl')
allatomsrelax = pd.read_pickle('allatomsrelax.pkl')
AFLOWtherm = pd.read_pickle('AFLOWtherm.pkl')

In [ ]:
ATMDATA={}
ATMDATA['ATM']=ATM
ATMDATA['ATMR']=ATMR
ATMDATA['ATME']=ATME
ATMDATA['ATMER']=ATMER
ATMDATA['ATMNE']=ATMNE
ATMDATA['ATMNER']=ATMNER

In [ ]:
mlcpaflow = np.load('mlcpdatafixed.npy',allow_pickle='TRUE').item()
mlcpaflowrelax = np.load('mlcpdatafixedrelaxed.npy',allow_pickle='TRUE').item()

In [ ]:
mlcpaflowresults={}
mlcpaflowrelaxresults={}
for i,j in zip(ATMDATA['ATM']['compound'],ATMDATA['ATM']['ICSD']):
    mlcpaflowresults[i+'_'+j]=mlcpaflow[i+'_'+j]
for i,j in zip(ATMDATA['ATMR']['compound'],ATMDATA['ATMR']['ICSD']):
    mlcpaflowrelaxresults[i+'_'+j]=mlcpaflowrelax[i+'_'+j]

In [ ]:
for i in ['ATM','ATME','ATMNE','ATMR','ATMER','ATMNER']:
    ATMDATA[i+'M']=ATMDATA[i].query('Etype==0')
    ATMDATA[i+'S']=ATMDATA[i].query('Etype==1')
    ATMDATA[i+'I']=ATMDATA[i].query('Etype==2')
for i in ['ATM','ATME','ATMNE','ATMR','ATMER','ATMNER']:
    ATMDATA[i+'2']=ATMDATA[i].query('nspecies<=2')
    ATMDATA[i+'M2']=ATMDATA[i+'M'].query('nspecies<=2')
    ATMDATA[i+'S2']=ATMDATA[i+'S'].query('nspecies<=2')
    ATMDATA[i+'I2']=ATMDATA[i+'I'].query('nspecies<=2')
kthreshold=15
for i in ['ATM','ATME','ATMR','ATMER']:
    ATMDATA[i+'U']=ATMDATA[i].query(f'kappa<={kthreshold}')
    ATMDATA[i+'MU']=ATMDATA[i+'M'].query(f'kappa<={kthreshold}')
    ATMDATA[i+'2U']=ATMDATA[i+'2'].query(f'kappa<={kthreshold}')
    ATMDATA[i+'M2U']=ATMDATA[i+'M2'].query(f'kappa<={kthreshold}')
for i in ['ATM','ATME','ATMR','ATMER']:
    ATMDATA[i+'B']=ATMDATA[i].query('0<=bias<=0.01')
    ATMDATA[i+'UB']=ATMDATA[i+'U'].query('0<=bias<=0.01')

In [ ]:
ATMDATA.keys()

In [ ]:
#each slice and its count
for i in ATMDATA.keys():
    print(i,len(ATMDATA[i]))

ATM: Base Data

E/NE: Elements Only, Not Elements Only

R: Relaxed

M/S/I: Metal, Semiconductor, Insulator

2: Binaries Only

U: Kappa's under 15 only

B: no +/- bias

## DESCRIBING THE DATASETS

In [ ]:
plt.figure(figsize=(6, 4))
plt.hist(kpred['ATM']['totalcp'],bins=100)
plt.title('Distribution of Total CP Values')
plt.xlabel('Total CP (GPa)')
plt.ylabel('Count')
plt.savefig('IMAGES/Distribution of Total CP Values.svg', bbox_inches='tight', dpi=1200)
plt.show()

In [ ]:
plt.hist(ATMDATA['ATMR']['rootkappa'],100);
#plt.title('Thermal Conductivity Distribution (AFLOW data)')
plt.xlabel('$\mathregular{\kappa^{1/2}}$ $\mathregular{(W/mK)^{1/2}}$')
plt.ylabel('count')
plt.savefig('IMAGES/Thermal Conductivity Distribution (AFLOW data).png', bbox_inches='tight', dpi=1200)

In [ ]:
plt.hist(ATMDATA['ATMR']['rootkappa'],np.linspace(0,20,100),alpha=0.4, label=f'All; n={len(ATMDATA['ATMR'])}');
plt.hist(ATMDATA['ATMRM']['rootkappa'],np.linspace(0,20,100),alpha=0.4, label=f'Egap=0 eV; n={len(ATMDATA['ATMRM'])}');
plt.hist(ATMDATA['ATMRI']['rootkappa'],np.linspace(0,20,100),alpha=0.4, label=f'Egap>0.5 eV; n={len(ATMDATA['ATMRI'])}');
plt.hist(ATMDATA['ATMRS']['rootkappa'],np.linspace(0,20,100),alpha=0.4, label=f'Egap<0.5 eV; n={len(ATMDATA['ATMRS'])}');

plt.legend()
plt.title('Thermal Conductivity Distribution by Band Gap Threshold')
plt.xlabel('thermal $\mathregular{conductivity^{1/2}}$ $\mathregular{(W/mK)^{1/2}}$')
plt.ylabel('count')
plt.xlim(0,10)
plt.savefig('IMAGES/Thermal Conductivity Distribution by Band Gap Threshold.png', bbox_inches='tight', dpi=1200)

In [ ]:
geometry_diff=[]
geometry_ratio=[]
length_ratio=[]
angle_ratio=[]
for i in range(len(AFLOWtherm['geometry'])):
    diffs=[]
    for j in range(6):
        if AFLOWtherm['geometry'][i][j]==0 and AFLOWtherm['real_geometry'][i][j]==0:
            diffs.append(1)
        else:
            diffs.append(AFLOWtherm['geometry'][i][j]/AFLOWtherm['real_geometry'][i][j])
    geometry_diff.append(diffs)
    geometry_ratio.append(np.mean(diffs))
    length_ratio.append(np.mean(diffs[:3]))
    angle_ratio.append(np.mean(diffs[3:]))

In [ ]:
plt.figure(figsize=(6, 4))
plt.hist(geometry_ratio,bins=1000,alpha=0.4,label='overall')
plt.hist(length_ratio,bins=1000,alpha=0.4,label='abc')
plt.hist(angle_ratio,bins=1000,alpha=0.4,label='angles')
plt.legend()
plt.title('Deviations of Lattice Parameters (Relaxed vs Original)')
plt.xlim(0.9,1.15)
plt.ylim(0,500)
plt.xlabel('Ratio of Lattice Parameter')
plt.ylabel('Count')
plt.savefig('IMAGES/Deviations of Lattice Parameters (Relaxed vs Original).png', bbox_inches='tight', dpi=1200)
plt.show()

In [ ]:
svalues=[]
pvalues=[]
dvalues=[]
fvalues=[]
gvalues=[]
for i,j in zip(ATMDATA['ATM']['compound'],ATMDATA['ATM']['ICSD']):
    struct=mlcpaflowresults[i+'_'+j]
    for k in range(len(struct['CP'])):
        svalues.append(struct['Xs'][k])
        pvalues.append(struct['Xp'][k])
        dvalues.append(struct['Xd'][k])
        fvalues.append(struct['Xf'][k])
        gvalues.append(struct['Xg'][k])

rsvalues=[]
rpvalues=[]
rdvalues=[]
rfvalues=[]
rgvalues=[]
for i,j in zip(ATMDATA['ATMR']['compound'],ATMDATA['ATMR']['ICSD']):
    struct=mlcpaflowrelaxresults[i+'_'+j]
    for k in range(len(struct['CP'])):
        rsvalues.append(struct['Xs'][k])
        rpvalues.append(struct['Xp'][k])
        rdvalues.append(struct['Xd'][k])
        rfvalues.append(struct['Xf'][k])
        rgvalues.append(struct['Xg'][k])

In [ ]:
len(ATMDATA['ATMEMU'])

## MLCP GRAPHS

In [ ]:
plt.figure(figsize=(8, 6), dpi=1200)
plt.scatter(ATMDATA['ATMEMU']['mmXs'],ATMDATA['ATMEMU']['mmcpcp'],alpha=0.10,c=ATMDATA['ATMEMU']['rootkappa'])
cbar=plt.colorbar(label='$\mathregular{\kappa^{1/2}}$ $\mathregular{(W/mK)^{1/2}}$')
cbar.solids.set(alpha=0.4)
plt.ylim(-800,250)
plt.xlabel('$\mathregular{\chi_s}$ of $\mathregular{(CP*\chi_s)_{min}}$ site')
plt.ylabel('CP of $\mathregular{(CP*\chi_s)_{min}}$ site (GPa)')
plt.savefig('IMAGES/AFLOWmmXsmmcpcp.png', bbox_inches='tight', dpi=1200)
plt.show

In [ ]:
fig = px.scatter(ATMDATA['ATMEMU'], x="mmXs", y="mmcpcp", color="rootkappa",title="meminCP and memXe all atoms",
                width=1000, height=600,hover_data=['compound','kappa','spacegroup'],opacity=0.25)
fig.update_traces(marker_size=10)
fig.show()

In [ ]:
plt.hist(ATMDATA['ATMEMU']['rootkappa'],np.linspace(0,15,94));
plt.hist(ATMDATA['ATMEM']['rootkappa'],np.linspace(0,15,94),alpha=0.4);
plt.title('things are cut off on the right tail')
plt.suptitle('distribution of the square root of thermal conductivity')

plt.xlabel('thermal conductivity^0.5')
plt.ylabel('count')

In [ ]:
print('% with both a + and - CP (original):', list(ATMDATA['ATMEMU']['bias']).count(0)/len(ATMDATA['ATMEM']['bias']))
print('% with both a + and - CP (relaxed):', list(ATMDATA['ATMERMU']['bias']).count(0)/len(ATMDATA['ATMERM']['bias']))

In [ ]:
plt.hist([ATMDATA['ATM']['negpctcp'],ATMDATA['ATMR']['negpctcp']],50,label=['Original Lattice','Relaxed Lattice']);
plt.title('Bias in Net CP values by Geometry Input')
plt.xlabel('Portion of Sites with Negative Net CP')
plt.ylabel('Count')
plt.legend()
plt.savefig('IMAGES/Bias in Net CP values by Geometry Input.png', bbox_inches='tight', dpi=1200)
plt.show

In [ ]:
plt.hist([ATMDATA['ATM'].query('0<=bias<=0.01')['negpctcp'],ATMDATA['ATMR'].query('0<=bias<=0.01')['negpctcp']],50,label=['Original Lattice','Relaxed Lattice']);
plt.title('Bias in Net CP values by Geometry Input, No Extremes')
plt.xlabel('Portion of Sites with Negative Net CP')
plt.ylabel('Count')
plt.savefig('IMAGES/Bias in Net CP values by Geometry Input No Extremes.png', bbox_inches='tight', dpi=1200)
plt.legend()

## PERCENTILE DESCRIPTORS

In [ ]:
ATMERM2_analysis=ATMDATA['ATMERMU'].copy()

In [ ]:
np.median(ATMDATA['ATMERMU']['kappa'])

In [ ]:
3.596066080285486/(len(ATMDATA['ATMERMU'])/10)**0.5

In [ ]:
4.573029413718083+2.576*0.21267702955642334

In [ ]:
CPvariables2=ATMERM2_analysis.columns

In [ ]:
CPvariables2[-70:]

In [ ]:
CPvariables=['totalcp', 'mincp', 'maxcp', 'rangecp', 'sumavgcp', 'stdevcp', 'sumcp',
       'Xs', 'avgcp', 'mincpavg', 'mincpdiff', 'mincpz', 'mmincp',
       'mmaxcp', 'mrangecp', 'msumavgcp', 'mstdevcp', 'msumcp', 'mmXs',
       'mavgcp', 'mmcpavg', 'mmcpcp', 'memincp', 'mesumavgcp', 'mesumcp', 'memXe',
       'negpctcp', 'bias', 'minXs', 'maxXs', 'rangeXs', 'avgXs', 'stdevXs',
       'minXp', 'maxXp', 'rangeXp', 'avgXp', 'stdevXp', 'minXd', 'maxXd',
       'rangeXd', 'avgXd', 'stdevXd', 'minXf', 'maxXf', 'rangeXf', 'avgXf',
       'stdevXf', 'minXg', 'maxXg', 'rangeXg', 'avgXg', 'stdevXg']

In [ ]:
CP_variable_types={}
CP_variable_types['CPmin']=['mincp','Xs','mincpavg','mincpz','mincpdiff']
CP_variable_types['CPmax']=['maxcp','mmaxcp']
CP_variable_types['CPmultmin']=['mmincp','mmcpcp', 'mmXs', 'mmcpavg']
CP_variable_types['CPXs']=['minXs','maxXs', 'rangeXs', 'avgXs', 'stdevXs']
CP_variable_types['CPXp']=['minXp', 'maxXp', 'rangeXp','avgXp', 'stdevXp']
CP_variable_types['CPXd']=['minXd', 'maxXd', 'rangeXd', 'avgXd', 'stdevXd']
CP_variable_types['CPXf']=['minXf', 'maxXf', 'rangeXf', 'avgXf', 'stdevXf']
CP_variable_types['CPXg']=['minXg', 'maxXg', 'rangeXg', 'avgXg', 'stdevXg']
CP_variable_types['CPminXe']=['memincp', 'memXe', 'mesumcp', 'mesumavgcp']
CP_variable_types['CPdistribution']=['stdevcp','mstdevcp','rangecp','mrangecp','negpctcp','bias']
CP_variable_types['CPoverall']=['totalcp', 'sumavgcp','sumcp','avgcp']
CP_variable_types['CPmultoverall']=['msumavgcp','msumcp','mavgcp']
#CP_variable_types['CPbasic']=[,]
#CP_variable_types['CPdistribution']=['negpctcp','bias']
#CP_variable_types['ML-CP Values']=['mincp','maxcp','sumcp','maxXs']

In [ ]:
for i in CPvariables:
    minimum=min(ATMERM2_analysis[i])
    maximum=max(ATMERM2_analysis[i])
    therange=maximum-minimum
    scaledlist=[]
    for j in ATMERM2_analysis[i]:
        value=(j-minimum)/therange
        scaledlist.append(value)
    ATMERM2_analysis[i+'scaled']=scaledlist

In [ ]:
ATMERM2scaledsections={}
ATMERM2scaledmeans={}
ATMERM2scaledmedians={}
ATMERM2scaledstdevs={}
ATMERM2scaledcounts={}
valuetierscaled=np.linspace(0.1,1,10)

for j in CPvariables:
    ATMERM2scaledmeans[j]=[]
    ATMERM2scaledstdevs[j]=[]
    ATMERM2scaledmedians[j]=[]
    ATMERM2scaledcounts[j]=[]

for i in range(len(valuetierscaled)):
    for j in CPvariables:
        ATMERM2scaledsections['under'+j+str(i)]=ATMERM2_analysis.query(f'{valuetierscaled[i-1] if i>0 else 0}<={j}scaled<={valuetierscaled[i]}')
for j in CPvariables:
    for i in range(len(valuetierscaled)):
        ATMERM2scaledmeans[j].append(np.mean(ATMERM2scaledsections['under'+j+str(i)]['kappa'].to_list()))
        ATMERM2scaledmedians[j].append(np.median(ATMERM2scaledsections['under'+j+str(i)]['kappa'].to_list()))
        ATMERM2scaledstdevs[j].append(np.std(ATMERM2scaledsections['under'+j+str(i)]['kappa'].to_list()))
        ATMERM2scaledcounts[j].append(len(ATMERM2scaledsections['under'+j+str(i)]['kappa']))

In [ ]:
ATMERM2pctsections={}
ATMERM2pctthresholds={}
ATMERM2pctmeans={}
ATMERM2pctmedians={}
ATMERM2pctstdevs={}
valuetierpct=np.linspace(0.1,1,10)

for j in CPvariables:
    ATMERM2pctmeans[j]=[]
    ATMERM2pctstdevs[j]=[]
    ATMERM2pctthresholds[j]=[]
    ATMERM2pctmedians[j]=[]

for j in CPvariables:
    for i in valuetierpct:
        ATMERM2pctthresholds[j].append(ATMERM2_analysis[j].quantile(i))
for j in CPvariables:
    for i in range(len(ATMERM2pctthresholds[j])):
        ATMERM2pctsections['under'+j+str(i)]=ATMERM2_analysis.query(f'{ATMERM2pctthresholds[j][i-1] if i>0 else min(ATMERM2_analysis[j])}<={j}<={ATMERM2pctthresholds[j][i]}')
for j in CPvariables:
    for i in range(len(valuetierpct)):
        ATMERM2pctmeans[j].append(np.mean(ATMERM2pctsections['under'+j+str(i)]['kappa'].to_list()))
        ATMERM2pctmedians[j].append(np.median(ATMERM2pctsections['under'+j+str(i)]['kappa'].to_list()))
        ATMERM2pctstdevs[j].append(np.std(ATMERM2pctsections['under'+j+str(i)]['kappa'].to_list()))

In [ ]:
ATMERM2threshsections={}
ATMERM2threshthresholds={}
ATMERM2threshmeans={}
ATMERM2threshmedians={}
ATMERM2threshstdevs={}
valuetierthresh=np.linspace(0.02,1,50)

for j in CPvariables:
    ATMERM2threshmeans[j]=[]
    ATMERM2threshstdevs[j]=[]
    ATMERM2threshthresholds[j]=[]
    ATMERM2threshmedians[j]=[]

for j in CPvariables:
    for i in valuetierthresh:
        ATMERM2threshthresholds[j].append(ATMERM2_analysis[j].quantile(i))
for j in CPvariables:
    for i in range(len(ATMERM2threshthresholds[j])):
        ATMERM2threshsections['under'+j+str(i)]=ATMERM2_analysis.query(f'{j}<={ATMERM2threshthresholds[j][i]}')
for j in CPvariables:
    for i in range(len(valuetierthresh)):
        ATMERM2threshmeans[j].append(np.mean(ATMERM2threshsections['under'+j+str(i)]['kappa'].to_list()))
        ATMERM2threshmedians[j].append(np.median(ATMERM2threshsections['under'+j+str(i)]['kappa'].to_list()))
        ATMERM2threshstdevs[j].append(np.std(ATMERM2threshsections['under'+j+str(i)]['kappa'].to_list()))

In [ ]:
for graphs in CP_variable_types.keys():
    plt.figure(figsize=(8, 6))
    plt.title(f'CP Descriptor Percentiles: {graphs} and Thermal Conductivity')
    for j in CP_variable_types[graphs]:
        plt.plot(valuetierpct,ATMERM2pctmeans[j],'o:',label=j)
        #plt.errorbar(valuetierpct,ATMERM2pctmeans[j],yerr=ATMERM2pctstdevs[j],linestyle='None')
    plt.xlabel('CP descriptor percentile')
    plt.ylabel('average thermal conductivity')
    plt.legend()
    plt.show()

In [ ]:
fig, axs = plt.subplots(4, 3,sharey=True, sharex=True, figsize=(16, 16))
colornumb=0
# CP_variable_types['CPmin']=['mincp','Xs','mincpavg','mincpz','mincpdiff']
# CP_variable_types['CPmax']=['maxcp','mmaxcp']
# CP_variable_types['CPmultmin']=['mmincp','mmcpcp', 'mmXs', 'mmcpavg']
# CP_variable_types['CPXs']=['minXs','maxXs', 'rangeXs', 'avgXs', 'stdevXs']
# CP_variable_types['CPXp']=['minXp', 'maxXp', 'rangeXp','avgXp', 'stdevXp']
# CP_variable_types['CPXd']=['minXd', 'maxXd', 'rangeXd', 'avgXd', 'stdevXd']
# CP_variable_types['CPXf']=['minXf', 'maxXf', 'rangeXf', 'avgXf', 'stdevXf']
# CP_variable_types['CPXg']=['minXg', 'maxXg', 'rangeXg', 'avgXg', 'stdevXg']
# CP_variable_types['CPminXe']=['memincp', 'memXe', 'mesumcp', 'mesumavg']
# CP_variable_types['CPdistribution']=['stdevcp','mstdevcp','rangecp','mrangecp','negpctcp','bias']
# CP_variable_types['CPoverall']=['totalcp', 'sumavgcp','sumcp','avgcp']
# CP_variable_types['CPmultoverall']=['msumavgcp','msumcp','mavgcp']

CP_variable_list=[['CPoverall','CPmin','CPdistribution'],['CPmultoverall','CPmultmin','CPmax'],['CPminXe','CPXs','CPXp'],['CPXd','CPXf','CPXg']]

for i in range(4):
    for j in range(3):
        axs[i, j].set_title(f'{CP_variable_list[i][j]}')
        for k in CP_variable_types[CP_variable_list[i][j]]:
            axs[i, j].plot(valuetierpct,ATMERM2pctmeans[k],'o:',label=k)
        axs[i, j].set_xlabel('CP descriptor percentile')
        axs[i, j].plot([0,1],[4.025173385580736,4.025173385580736],':',c='k')
        axs[i, j].plot([0,1],[5.12088544185543,5.12088544185543],':',c='k')
        axs[i, j].legend()
        axs[i, j].set_ylabel('median thermal conductivity')
for ax in axs.flat:
    ax.label_outer()
plt.savefig('IMAGES/AFLOWVarmean.png', bbox_inches='tight', dpi=1500)

In [ ]:
fig, axs = plt.subplots(4, 3,sharey=True, sharex=True, figsize=(16, 16))
colornumb=0
# CP_variable_types['CPmin']=['mincp','Xs','mincpavg','mincpz','mincpdiff']
# CP_variable_types['CPmax']=['maxcp','mmaxcp']
# CP_variable_types['CPmultmin']=['mmincp','mmcpcp', 'mmXs', 'mmcpavg']
# CP_variable_types['CPXs']=['minXs','maxXs', 'rangeXs', 'avgXs', 'stdevXs']
# CP_variable_types['CPXp']=['minXp', 'maxXp', 'rangeXp','avgXp', 'stdevXp']
# CP_variable_types['CPXd']=['minXd', 'maxXd', 'rangeXd', 'avgXd', 'stdevXd']
# CP_variable_types['CPXf']=['minXf', 'maxXf', 'rangeXf', 'avgXf', 'stdevXf']
# CP_variable_types['CPXg']=['minXg', 'maxXg', 'rangeXg', 'avgXg', 'stdevXg']
# CP_variable_types['CPminXe']=['memincp', 'memXe', 'mesumcp', 'mesumavg']
# CP_variable_types['CPdistribution']=['stdevcp','mstdevcp','rangecp','mrangecp','negpctcp','bias']
# CP_variable_types['CPoverall']=['totalcp', 'sumavgcp','sumcp','avgcp']
# CP_variable_types['CPmultoverall']=['msumavgcp','msumcp','mavgcp']

CP_variable_list=[['CPoverall','CPmin','CPdistribution'],['CPmultoverall','CPmultmin','CPmax'],['CPminXe','CPXs','CPXp'],['CPXd','CPXf','CPXg']]

for i in range(4):
    for j in range(3):
        axs[i, j].set_title(f'{CP_variable_list[i][j]}')
        for k in CP_variable_types[CP_variable_list[i][j]]:
            axs[i, j].plot(valuetierpct,ATMERM2pctmedians[k],'o:',label=k)
        axs[i, j].set_xlabel('CP descriptor percentile')
        axs[i, j].legend()
        axs[i, j].set_ylabel('median thermal conductivity')
for ax in axs.flat:
    ax.label_outer()
plt.savefig('IMAGES/AFLOWVar.png', bbox_inches='tight', dpi=1500)

In [ ]:
for graphs in CP_variable_types.keys():
    plt.figure(figsize=(8, 6))
    plt.title(f'CP Descriptor Percentiles: {graphs} and Thermal Conductivity')
    for j in CP_variable_types[graphs]:
        plt.plot(valuetierpct,ATMERM2pctmedians[j],'o:',label=j)
        #plt.errorbar(valuetierpct,ATMERM2pctmedians[j],yerr=ATMERM2pctstdevs[j],linestyle='None')
    plt.xlabel('CP descriptor percentile')
    plt.ylabel('median thermal conductivity')
    plt.legend()
    plt.show()

In [ ]:
for graphs in CP_variable_types.keys():
    plt.figure(figsize=(8, 6))
    plt.title(f'CP Descriptor Scaled: {graphs} and thermal conductivity')
    for j in CP_variable_types[graphs]:
        plt.plot(valuetierscaled,ATMERM2scaledmeans[j],'o:',label=j)
        #plt.errorbar(valuetierscaled,ATMERM2scaledmeans[j],yerr=ATMERM2scaledstdevs[j],linestyle='None')
        plt.errorbar(valuetierscaled,ATMERM2scaledmeans[j],yerr=[5/a**0.5 if a!=0 else 6 for a in ATMERM2scaledcounts[j]],linestyle='None')
    
    plt.xlabel('CP descriptor scaled')
    plt.ylabel('average thermal conductivity')
    plt.legend()
    plt.show()

In [ ]:
for graphs in CP_variable_types.keys():
    plt.figure(figsize=(8, 6))
    plt.title(f'CP Descriptor Scaled: {graphs} and thermal conductivity')
    for j in CP_variable_types[graphs]:
        plt.plot(valuetierscaled,ATMERM2scaledmedians[j],'o:',label=j)
        #plt.errorbar(valuetierscaled,ATMERM2scaledmedians[j],yerr=ATMERM2scaledstdevs[j],linestyle='None')
        plt.errorbar(valuetierscaled,ATMERM2scaledmedians[j],yerr=[5/a**0.5 if a!=0 else 6 for a in ATMERM2scaledcounts[j]],linestyle='None')
    
    plt.xlabel('CP descriptor scaled')
    plt.ylabel('average thermal conductivity')
    plt.legend()
    plt.show()

## INFORMATION ON ALL ATOMS

In [ ]:
allatoms.columns

In [ ]:
allatomsnumbers=allatoms.drop(['formulas','ismin','site', 'atom','kunder'], axis=1)

In [ ]:
allatomsnumbers=allatomsnumbers.query('kappa<15')

In [ ]:
fig = px.scatter(allatomsnumbers, x="Xs", y="cp", color="rootkappa",title="",
                width=1000, height=600,hover_data=['mult'],opacity=0.07)
fig.update_traces(marker_size=10)
fig.write_image("IMAGES/AFLOWallatomsXsCP.png")
fig.show()

## CORRELATIONS

In [ ]:
corr = allatomsnumbers.corr(method = 'pearson')
mask = np.triu(np.ones_like(corr, dtype=bool))
cmap = sns.diverging_palette(230, 20, as_cmap=True)
plt.figure(figsize=(10,8), dpi =500)
sns.heatmap(corr, mask=mask, cmap=cmap, vmax=.3, center=0,
            square=True, linewidths=.5, annot=True, fmt=".2f", cbar_kws={"shrink": .5})
plt.savefig('IMAGES/Correlation AFLOWallatoms.png', bbox_inches='tight', dpi=1200)
plt.show()

In [ ]:
AFLOWthermmlcpnumbers.columns

In [ ]:
AFLOWthermmlcpnumbers=ATMDATA['ATMU'].drop(['compound', 'real_spacegroup', 'compoundtype', 'ICSD','positions', 'prototype', 'spacegroup', 'Pearson',
       'geometry', 'real_geometry', 'species', 'nspecies', 'Wletters',
       'real_Wletters', 'Wmultiplicities', 'real_Wmultiplicities','auid', 'aurl', 'sites',
       'real_sites','kunder','natoms', 'Egap',
       'Wsite_symmetries', 'summult', 'real_summult',
       'real_multiplier', 'multiplier', 'issues', 'reduced','site', 'atom','mmsite',
       'mmatom','kclass','bulk_modulus','debye','Egap_type','gruneisen', 'enthalpy', 'energy', 'speed_sound', 'lenpos',
       'temperature','Egap_fit', 'Etype', 'fixedsites', 'sitecount',
       'fixedmultiplier', 'fixedrealsites', 'realsitecount',
       'fixedrealmultiplier', 'relaxissues','Xs2','4rootkappa', 'ktiers','memsite', 'mematom'], axis=1)
corr = AFLOWthermmlcpnumbers.corr(method = 'pearson')
mask = np.triu(np.ones_like(corr, dtype=bool))
f, ax = plt.subplots(figsize=(11, 9))

# Generate a custom diverging colormap
cmap = sns.diverging_palette(230, 20, as_cmap=True)
sns.set(font_scale=0.65)
sns.set_style("whitegrid", {'axes.grid' : False})
# Draw the heatmap with the mask and correct aspect ratio
sns.heatmap(corr, mask=mask, cmap=cmap, vmax=.3, center=0,
            square=True, linewidths=.5, cbar_kws={"shrink": .5})
# plt.figure(figsize=(10,8), dpi =500)
# sns.heatmap(corr,annot=True,fmt=".2f", linewidth=.5)
plt.savefig('IMAGES/Correlation AFLOW.png', bbox_inches='tight', dpi=2000)
plt.show()

In [ ]:
ATMDATA.keys()

## MACHINE LEARNING

In [ ]:
kpred={}
for i in ATMDATA.keys():
    kpred[i]=ATMDATA[i]
    print(i+' '+str(len(ATMDATA[i])))

In [ ]:
#data slices used for ML
full_list_ML=['ATM', 'ATMR', 'ATME', 'ATMER', 'ATMNE', 'ATMNER', 'ATMM', 'ATMS', 'ATMI', 'ATMEM', 'ATMES', 'ATMEI', 'ATMNEM', 'ATMNES', 'ATMNEI', 'ATMRM', 'ATMRS', 'ATMRI', 'ATMERM', 'ATMERS', 'ATMERI', 'ATMNERM', 'ATMNERS', 'ATMNERI', 'ATM2', 'ATMM2', 'ATMS2', 'ATMI2', 'ATME2', 'ATMEM2', 'ATMES2', 'ATMEI2', 'ATMNE2', 'ATMNEM2', 'ATMNES2', 'ATMNEI2', 'ATMR2', 'ATMRM2', 'ATMRS2', 'ATMRI2', 'ATMER2', 'ATMERM2', 'ATMERS2', 'ATMERI2', 'ATMNER2', 'ATMNERM2', 'ATMNERS2', 'ATMNERI2', 'ATMU', 'ATMMU', 'ATM2U', 'ATMM2U', 'ATMEU', 'ATMEMU', 'ATME2U', 'ATMEM2U', 'ATMRU', 'ATMRMU', 'ATMR2U', 'ATMRM2U', 'ATMERU', 'ATMERMU', 'ATMER2U', 'ATMERM2U','ATMB', 'ATMEB', 'ATMRB', 'ATMERB','ATMUB', 'ATMEUB', 'ATMRUB', 'ATMERUB']

In [ ]:
#various combinations of the slices, can be examined individually
NE_list=[x for x in full_list_ML if 'NE' in x] #has NE
Use_list=[x for x in full_list_ML if x not in NE_list] #does not have NE
Orig_list=[x for x in full_list_ML if 'R' not in x] #original geo
R_list=[x for x in full_list_ML if 'R' in x] #relaxed geo
E_list=[x for x in Use_list if 'E' in x] #has E
AllE_list=[x for x in Use_list if 'E' not in x] #no element restrictions
All2_list=[x for x in Use_list if '2' in x] #binaries
AllE23_list=[x for x in Use_list if '2' not in x] #not binaries
M_list=[x for x in Use_list if 'M' in x[3:]] #no band gap
S_list=[x for x in Use_list if 'S' in x[3:]] #small band gap
I_list=[x for x in Use_list if 'I' in x[3:]] #larger band gap
AllT_list=[x for x in Use_list if 'I' not in x and 'S' not in x and 'M' not in x[3:]] #no band gap restriction
TM_list=AllT_list+M_list #no restriction on band gap, or metals only
U_list=[x for x in full_list_ML if 'U' in x] #under 15 W/mK kappa
TMU_list=[x for x in TM_list if x in U_list] #no restriction on band gap, or metals only with under 15 W/mK kappa
TMAll_list=[x for x in TM_list if x not in U_list] #no under 15 W/mK kappa restriction
B_list=[x for x in full_list_ML if 'B' in x] #bias filtered

In [ ]:
#writing features for everything
for i in full_list_ML:
    print(i)
    kpred[i+'features']=ref3.write_features_ratio(kpred[i],label_col = 'compound')

In [ ]:
#joining datasets
for i in TM_list:
    kpred[i+'data']=kpred[i+'features'].join(kpred[i].reset_index(drop=True))

In [ ]:
#splitting for classifier
threshold=3
for i in TM_list:
    kpred[i+'data']['under'+str(threshold)]=[1 if k<threshold else 0 for k in kpred[i+'data']['kappa']]

for i in TM_list:
    kpred['train'+i],kpred['ytrain'+i],kpred['test'+i],kpred['ytest'+i]=ref3.split(kpred[i+'data'],kpred[i+'data']['under'+str(threshold)],split_fraction=0.8,rand_state=5)


In [ ]:
#splitting for Regressor
for i in TM_list:
    kpred['Rtrain'+i],kpred['Rytrain'+i],kpred['Rtest'+i],kpred['Rytest'+i]=ref3.split(kpred[i+'data'],kpred[i+'data']['kappa'],split_fraction=0.8,rand_state=5)


In [ ]:
#splitting for root Regressor
for i in TM_list:
    kpred['R2train'+i],kpred['R2ytrain'+i],kpred['R2test'+i],kpred['R2ytest'+i]=ref3.split(kpred[i+'data'],kpred[i+'data']['rootkappa'],split_fraction=0.8,rand_state=5)

0:217 is standard composition features

217:221 is composition and other identifiers

221 is kappa

222:228 is other numerical qualities

228:248 is other structural detail

248:252 is Egap data

252:254 is AFLOW identifiers

254:264 is structural and sites data and set filters

264:269 is other versions of kappa

269 is reduced formula

270:324 is mlcp numbers

324:330 is atoms/sites

330 is the threshold value binary

In [ ]:
#checking for missing data
for i in kpred['ATMUdata'].describe().columns:
    if kpred['ATMUdata'].describe().iloc[0].loc[i]<len(kpred['ATMUdata']):
        print(i)
    else:
        continue

In [ ]:
#creating the training data
compfeats= np.r_[0:217]
mlcpfeats= np.r_[270:324]
bothfeats= np.r_[0:217,270:324]
for i in TM_list:
    kpred['Xtrain'+i]=kpred['train'+i].iloc[:,compfeats]
    kpred['Xtrain'+i+'CP']=kpred['train'+i].iloc[:,bothfeats]
    kpred['Xtrain'+i+'CPO']=kpred['train'+i].iloc[:,mlcpfeats]
    kpred['Xtest'+i]=kpred['test'+i].iloc[:,compfeats]
    kpred['Xtest'+i+'CP']=kpred['test'+i].iloc[:,bothfeats]
    kpred['Xtest'+i+'CPO']=kpred['test'+i].iloc[:,mlcpfeats]

for i in TM_list:
    kpred['RXtrain'+i]=kpred['Rtrain'+i].iloc[:,compfeats]
    kpred['RXtrain'+i+'CP']=kpred['Rtrain'+i].iloc[:,bothfeats]
    kpred['RXtrain'+i+'CPO']=kpred['Rtrain'+i].iloc[:,mlcpfeats]
    kpred['RXtest'+i]=kpred['Rtest'+i].iloc[:,compfeats]
    kpred['RXtest'+i+'CP']=kpred['Rtest'+i].iloc[:,bothfeats]
    kpred['RXtest'+i+'CPO']=kpred['Rtest'+i].iloc[:,mlcpfeats]

for i in TM_list:
    kpred['R2Xtrain'+i]=kpred['R2train'+i].iloc[:,compfeats]
    kpred['R2Xtrain'+i+'CP']=kpred['R2train'+i].iloc[:,bothfeats]
    kpred['R2Xtrain'+i+'CPO']=kpred['R2train'+i].iloc[:,mlcpfeats]
    kpred['R2Xtest'+i]=kpred['R2test'+i].iloc[:,compfeats]
    kpred['R2Xtest'+i+'CP']=kpred['R2test'+i].iloc[:,bothfeats]
    kpred['R2Xtest'+i+'CPO']=kpred['R2test'+i].iloc[:,mlcpfeats]

In [ ]:
#training the classifier models 
for i in TM_list:
    for j in ['','CP','CPO']:
        #kpred['rfcmodel'+i+j]=RandomForestClassifier(max_depth=3).fit(kpred['Xtrain'+i+j],kpred['ytrain'+i])
        #kpred['dtcmodel'+i+j]=DecisionTreeClassifier(max_depth=3).fit(kpred['Xtrain'+i+j],kpred['ytrain'+i])
        #kpred['hgbcmodel'+i+j]=HistGradientBoostingClassifier().fit(kpred['Xtrain'+i+j],kpred['ytrain'+i])
        kpred['gbcmodel'+i+j]=GradientBoostingClassifier(max_features=0.6).fit(kpred['Xtrain'+i+j],kpred['ytrain'+i])

In [ ]:
#testing the classifier models
for i in TM_list:
    for j in ['','CP','CPO']:
        gorup=i+j
        # ref3.rocconfusion(kpred['rfcmodel'+i+j],kpred['Xtest'+i+j],kpred['ytest'+i],[f"k>{threshold}", f"k<{threshold}"]],title=gorup)
        # ref3.rocconfusion(kpred['dtcmodel'+i+j],kpred['Xtest'+i+j],kpred['ytest'+i],[f"k>{threshold}", f"k<{threshold}"]],title=gorup)
        # ref3.rocconfusion(kpred['hgbcmodel'+i+j],kpred['Xtest'+i+j],kpred['ytest'+i],[f"k>{threshold}", f"k<{threshold}"]],title=gorup)
        ref3.rocconfusion(kpred['gbcmodel'+i+j],kpred['Xtest'+i+j],kpred['ytest'+i],[f"k>{threshold}", f"k<{threshold}"],title=gorup)

In [ ]:
#training the regressor models 
for i in TM_list:
    for j in ['','CP','CPO']:
        #kpred['rfrmodel'+i+j]=RandomForestRegressor(max_depth=3).fit(kpred['RXtrain'+i+j],kpred['Rytrain'+i])
        kpred['gbrmodel'+i+j]=GradientBoostingRegressor(max_features=0.6).fit(kpred['RXtrain'+i+j],kpred['Rytrain'+i])
        print(i+j)

In [ ]:
#training the root regressor models 
for i in TM_list:
    for j in ['','CP','CPO']:
        #kpred['rfrmodel'+i+j]=RandomForestRegressor(max_depth=3).fit(kpred['RXtrain'+i+j],kpred['Rytrain'+i])
        kpred['gbrmodel2'+i+j]=GradientBoostingRegressor(max_features=0.6).fit(kpred['R2Xtrain'+i+j],kpred['R2ytrain'+i])
        print(i+j)

In [ ]:
#testing the regressor models 
for i in TM_list:
    for j in ['','CP','CPO']:
        gorup=i+j
        if 'U' in i:
            dimvalue=15 
        else:
            dimvalue=40
        ref3.test_rf_modelboost3(kpred['RXtest'+i+j],kpred['Rytest'+i],kpred['gbrmodel'+i+j],dim=dimvalue, title=gorup)

In [ ]:
#testing the root regressor models 
for i in TM_list:
    for j in ['','CP','CPO']:
        gorup=i+j
        if 'U' in i:
            dimvalue=15
        else:
            dimvalue=40
        ref3.test_rf_modelboost3root(kpred['R2Xtest'+i+j],kpred['R2ytest'+i],kpred['gbrmodel2'+i+j],dim=dimvalue,title=gorup)

In [ ]:
#finding feature importance
for i in TM_list:
    for j in ['','CP','CPO']:
        gorup=i+j
        ref3.feature_importance(kpred['Xtrain'+i+j],kpred['gbcmodel'+i+j],n_features=15,title=' gorup')


looking at SG 204

In [ ]:
#sorting out this space group
index204={}
for i in TM_list:
    index204[i]=[x for x in kpred[i+'data'].query('spacegroup=="204"').index if x in kpred['Xtest'+i].index]

index204R={}
for i in TM_list:
    index204R[i]=[x for x in kpred[i+'data'].query('spacegroup=="204"').index if x in kpred['RXtest'+i].index]

index204R2={}
for i in TM_list:
    index204R2[i]=[x for x in kpred[i+'data'].query('spacegroup=="204"').index if x in kpred['R2Xtest'+i].index]

In [ ]:
for i in TM_list:
    for j in ['','CP','CPO']:
        gorup=i+j
        # ref3.rocconfusion(kpred['rfcmodel'+i+j],kpred['Xtest'+i+j],kpred['ytest'+i],[f"k>{threshold}", f"k<{threshold}"]],title=gorup)
        # ref3.rocconfusion(kpred['dtcmodel'+i+j],kpred['Xtest'+i+j],kpred['ytest'+i],[f"k>{threshold}", f"k<{threshold}"]],title=gorup)
        # ref3.rocconfusion(kpred['hgbcmodel'+i+j],kpred['Xtest'+i+j],kpred['ytest'+i],[f"k>{threshold}", f"k<{threshold}"]],title=gorup)
        ref3.rocconfusion(kpred['gbcmodel'+i+j],kpred['Xtest'+i+j].loc[index204[i]],kpred['ytest'+i].loc[index204[i]],[f"k>{threshold}", f"k<{threshold}"],title=gorup+'SG 204')

In [ ]:
TM_list204=[]
for i in TM_list:
    if len(kpred['Rytest'+i].loc[index204R[i]])>=10:
        TM_list204.append(i)
    else:
        continue

In [ ]:
for i in TM_list204:
    for j in ['','CP','CPO']:
        grupo=i+j
        #ref3.test_rf_model2(kpred['RXtrain'+i+j],kpred['Rytrain'+i],kpred['RXtest'+i+j],kpred['Rytest'+i],kpred['rfrmodel'+i+j],dim=40)
        ref3.test_rf_modelboost3(kpred['RXtest'+i+j].loc[index204R[i]],kpred['Rytest'+i].loc[index204R[i]],kpred['gbrmodel'+i+j],dim=20,title=grupo)

In [ ]:
TM_list2042=[]
for i in TM_list:
    if len(kpred['R2ytest'+i].loc[index204R2[i]])>=10:
        TM_list2042.append(i)
    else:
        continue

In [ ]:
for i in TM_list2042:
    for j in ['','CP','CPO']:
        grupo=i+j
        #ref3.test_rf_model2(kpred['RXtrain'+i+j],kpred['Rytrain'+i],kpred['RXtest'+i+j],kpred['Rytest'+i],kpred['rfrmodel'+i+j],dim=40)
        ref3.test_rf_modelboost3(kpred['R2Xtest'+i+j].loc[index204R2[i]],kpred['R2ytest'+i].loc[index204R2[i]],kpred['gbrmodel2'+i+j],dim=4,title=grupo)